# 🔴WAJIB RUNNING LIBRARY🔴

In [21]:
import json, re
import pandas as pd
import gspread
import os
import zipfile
import numpy as np
import getpass

from pathlib import Path
from google.auth.exceptions import GoogleAuthError
from datetime import datetime, timedelta, time
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build

SERVICE_ACCOUNT_FILE  = '/home/kes.ts23/infokes/iqbaalfadil/script/AKTIVASI/credential-sa-spreadsheetchecklist.json'
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
credentials           = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
gc                    = gspread.authorize(credentials)
service               = build("sheets", "v4", credentials=credentials)

# 🔴ISI DATA KONFIGURASI TERLEBIH DAHULU🔴

In [80]:
# Menentukan ID Tiket dan PIC
IDTIKET     = '#TS-2501'
# IDTIKET     = '#TS-1960'
PIC         = 'ts_iqbal_agus'
NO_MEMO     = '582/ADM-MEMO/OPR-RR/12/2025'
# NO_MEMO     = '532/ADM-MEMO/OPR-AR/11/2025'
PELANGGAN   = 'Lama'
CREATED_AT  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 🔴TAHAP 1🔴

☑️MENGAMBIL DATA INTIME☑️

In [90]:
# Use URL ID
# revisi dengan no  memo, dan id tiket untuk append tiket ke intime di proses ketika sudah sesuai
spreadsheet_id  = '1IAR5thTwLHs42eJ28sqJMPbslhHIm8HnJqH7iTVhDmc'
sheet           = gc.open_by_key(spreadsheet_id)
worksheet       = sheet.worksheet('Detail APJ')
data            = worksheet.get_all_values()

df                = pd.DataFrame(data[1:], columns=data[0])
nama_kolom_tiket  = 'TIKET'
df_intime       = df[df[nama_kolom_tiket] == IDTIKET]
df_intime

,ID PELANGGAN,ID JEJARING,PROVINSI,NAMA KOTA/KAB,NAMA PUSKESMAS,NAMA JEJARING,JENIS FASKES,ID TIKET,STATUS,APLIKASI,LAPOR TRX,ID REVISI,REVIEW TRX,APPROVED TRX,ID AKT,NOMOR MEMO,REQ AKTIVASI,PROSES AKT,SELESAI AKT,KETERANGAN,TIKET
26,P2171010101,PU930,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Lengkang,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,Ae36P2171010101PU930,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
27,P2171010101,PU931,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pulau Sarang,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,Aab6P2171010101PU931,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
28,P2171010101,PU932,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Mongkol,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,A03eP2171010101PU932,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
29,P2171010101,PU933,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pemping,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,A915P2171010101PU933,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
30,P2171010101,PU934,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pulau Labun,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,A440P2171010101PU934,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
31,P2171010101,PU935,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kepala Jeri,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,A84cP2171010101PU935,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
32,P2171010101,PU936,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kasu 1,PUSTU,,,EPUSTU,.,,4 Dec 2025 14:24:58,4 Dec 2025 14:30:58,Af56P2171010101PU936,582/ADM-MEMO/OPR-RR/12/2025,4 Dec 25,,,,#TS-2501
33,P2171010101,PU937,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kasu 2,PUSTU,,,EPUSTU,.,,04/12/2025,04/12/2025,A7deP2171010101PU937,582/ADM-MEMO/OPR-RR/12/2025,04/12/2025,,,,#TS-2501
34,P2171010101,PU938,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Bertam,PUSTU,,,EPUSTU,.,,04/12/2025,04/12/2025,A670P2171010101PU938,582/ADM-MEMO/OPR-RR/12/2025,04/12/2025,,,,#TS-2501
35,P2171010101,PU939,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pecong,PUSTU,,,EPUSTU,.,,04/12/2025,04/12/2025,A74eP2171010101PU939,582/ADM-MEMO/OPR-RR/12/2025,04/12/2025,,,,#TS-2501


☑️MENGAMBIL DATA MEMO GO LIVE☑️

In [94]:
# get data dengan no memo dan kolom selesai aktivasi yang kosong
spreadsheet_id  = '1IAR5thTwLHs42eJ28sqJMPbslhHIm8HnJqH7iTVhDmc'
sheet           = gc.open_by_key(spreadsheet_id)
worksheet       = sheet.worksheet('Memo Go Live')
data            = worksheet.get_all_values()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

df_memo            = pd.DataFrame(data[1:], columns=data[0])
coloumn_memo       = 'NOMOR MEMO'
coloumn_pelanggan  = 'Jenis Pelanggan'
df_nomemo          = df_memo[df_memo[coloumn_memo] == NO_MEMO]
df_pelanggan       = df_nomemo[df_nomemo[coloumn_pelanggan] == PELANGGAN]

print('Berikut data memo go live:')
df_pelanggan  

Berikut data memo go live:


,No,ID PELANGGAN,ID JEJARING,PROVINSI,NAMA KOTA/KAB,NAMA PUSKESMAS,NAMA JEJARING,JENIS FASKES,Jenis Pelanggan,NOMOR MEMO,STATUS
0,1,P2171010101,PU930,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Lengkang,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
1,2,P2171010101,PU931,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pulau Sarang,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
2,3,P2171010101,PU932,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Mongkol,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
3,4,P2171010101,PU933,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pemping,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
4,5,P2171010101,PU934,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pulau Labun,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
5,6,P2171010101,PU935,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kepala Jeri,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
6,7,P2171010101,PU936,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kasu 1,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
7,8,P2171010101,PU937,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Kasu 2,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
8,9,P2171010101,PU938,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Bertam,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,
9,10,P2171010101,PU939,Kepulauan Riau,Kota Batam,Belakang Padang,Pustu Pecong,Pustu,Lama,582/ADM-MEMO/OPR-RR/12/2025,


☑️COMPARE MEMO AKTIVASI DENGAN INTIME☑️

In [52]:
key_cols = [
    "ID PELANGGAN",
    "ID JEJARING",
    "PROVINSI",
    "NAMA KOTA/KAB",
    "NAMA PUSKESMAS",
    "NAMA JEJARING",
    "NOMOR MEMO"
]

In [97]:
df_nomemo_copy = df_nomemo.copy()
df_intime_copy = df_intime.copy()

# Normalisasi menjadi string
for df in [df_nomemo_copy, df_intime_copy]:
    df["ID PELANGGAN"]   = df["ID PELANGGAN"].astype(str).str.strip()
    df["ID JEJARING"]    = df["ID JEJARING"].astype(str).str.strip().str.upper()
    df["NAMA JEJARING"]  = df["NAMA JEJARING"].astype(str).str.strip()
    df["JENIS FASKES"]   = df["JENIS FASKES"].astype(str).str.strip()
    df["NAMA PUSKESMAS"] = df["NAMA PUSKESMAS"].astype(str).str.strip()
    df["PROVINSI"]       = df["PROVINSI"].astype(str).str.strip()


# REPORT 1 : SUMMARY JUMLAH DATA
print("REPORT 1 : SUMMARY TOTAL DATA")

total_memo   = len(df_nomemo_copy)
total_intime = len(df_intime_copy)

print(f"Total Data Memo   : {total_memo}")
print(f"Total Data Intime : {total_intime}")

if total_memo > total_intime:
    print(f"⚠️ WARNING: Data memo ({total_memo}) lebih banyak dari data intime ({total_intime})")
elif total_memo == total_intime:
    print(f"✅ Jumlah data memo dan intime sama ({total_memo})")
else:
    print(f"ℹ️ Data memo ({total_memo}) lebih sedikit dari data intime ({total_intime})")


# DUPLICATE MAP
def _build_dup_map(df: pd.DataFrame, subset_cols: list[str], label: str) -> dict:
    dmap    = {}
    dup_df  = df[df.duplicated(subset=subset_cols, keep=False)].copy()
    if dup_df.empty:
        return dmap

    # group key -> list sheet rows
    for _, g in dup_df.groupby(subset_cols, dropna=False):
        idxs        = g.index.tolist()
        sheet_rows  = sorted([i + 2 for i in idxs])
        rows_txt    = " dan ".join(map(str, sheet_rows)) if len(sheet_rows) == 2 else ", ".join(map(str, sheet_rows))

        msg = f"Duplikat {label}: baris no {rows_txt}"
        for i in idxs:
            dmap.setdefault(i, []).append(msg)

    return dmap

dup_map = {}
dup_a = _build_dup_map(df_nomemo_copy, ["ID PELANGGAN", "ID JEJARING"], "ID FASKES SAMA")
dup_b = _build_dup_map(df_nomemo_copy, ["NAMA JEJARING"], "NAMA JEJARING SAMA")
dup_c = _build_dup_map(df_nomemo_copy, ["ID PELANGGAN", "ID JEJARING", "NAMA JEJARING"], "ID FASKES, ID JEJARING DAN NAMA JEJARING SAMA")

# gabungkan ke satu map
for m in (dup_a, dup_b, dup_c):
    for k, v in m.items():
        dup_map.setdefault(k, []).extend(v)


# HELPER STATUS SEDERHANA
COMPARE_COLS = [
    ("ID PELANGGAN",   "id faskes beda"),
    ("ID JEJARING",    "id jejaring beda"),
    ("NAMA PUSKESMAS", "nama faskes beda"),
    ("JENIS FASKES", "jenis jejaring beda"),
    ("NAMA JEJARING",  "nama jejaring beda"),
    ("PROVINSI",       "provinsi beda"),
]

def norm(val):
    if pd.isna(val):
        return ""
    return str(val).strip().upper()


def build_status_simple(memo_row: pd.Series, intime_row: pd.Series | None) -> str:
    if intime_row is None:
        return "data tidak ada di intime"

    same_flags = []
    diffs = []

    for col, diff_label in COMPARE_COLS:
        same = (norm(memo_row[col]) == norm(intime_row[col]))
        same_flags.append(same)
        if not same:
            diffs.append(diff_label)

    if all(same_flags):
        return "sama"

    if not any(same_flags):
        return "data tidak ada di intime"

    return ", ".join(diffs)



# REPORT 2 : DETAIL PERBANDINGAN MEMO DENGAN INTIME 
print("\nREPORT 2 : DETAIL PERBANDINGAN MEMO DENGAN INTIME")

rows_output = []

for idx, row in df_nomemo_copy.iterrows():
    sheet_row = idx + 2

    # Matching baris menggunakan id pelanggan dan id jejaring
    cond = ((df_intime_copy["ID PELANGGAN"] == row["ID PELANGGAN"])
    & (df_intime_copy["ID JEJARING"] == row["ID JEJARING"]))

    intime_row = df_intime_copy.loc[cond].iloc[0] if cond.any() else None

    status_base = build_status_simple(row, intime_row)

    dup_text = " | ".join(dup_map.get(idx, []))
    final_status = status_base if not dup_text else f"{status_base} | {dup_text}"

    # Flag untuk sorting: beda/duplikat ditaruh paling atas
    is_issue = (status_base != "sama") or bool(dup_text)

    rows_output.append({
        "Row Sheet": sheet_row,

        "ID Memo": row["ID PELANGGAN"],
        "ID Intime": "" if intime_row is None else intime_row["ID PELANGGAN"],

        "ID Jejaring Memo": row["ID JEJARING"],
        "ID Jejaring Intime": "" if intime_row is None else intime_row["ID JEJARING"],

        "Nama Memo": row["NAMA PUSKESMAS"],
        "Nama Intime": "" if intime_row is None else intime_row["NAMA PUSKESMAS"],

        "Nama Jejaring Memo": row["NAMA JEJARING"],
        "Nama Jejaring Intime": "" if intime_row is None else intime_row["NAMA JEJARING"],


        "Jenis Faskes Memo": row["JENIS FASKES"],
        "Jenis Faskes Intime": "" if intime_row is None else intime_row["JENIS FASKES"],

        "Prov Memo": row["PROVINSI"],
        "Prov Intime": "" if intime_row is None else intime_row["PROVINSI"],

        "Status": final_status,
        "_is_issue": is_issue, 
    })

df_output = pd.DataFrame(rows_output)


# Fungsi shortir untuk data yang berbeda atau duplikat akan dimunculkan paling atas
df_output = (
    df_output
    .sort_values(by=["_is_issue", "Row Sheet"], ascending=[False, True])
    .drop(columns=["_is_issue"])
)

df_output


REPORT 1 : SUMMARY TOTAL DATA
Total Data Memo   : 91
Total Data Intime : 91
✅ Jumlah data memo dan intime sama (91)

REPORT 2 : DETAIL PERBANDINGAN MEMO DENGAN INTIME


,Row Sheet,ID Memo,ID Intime,ID Jejaring Memo,ID Jejaring Intime,Nama Memo,Nama Intime,Nama Jejaring Memo,Nama Jejaring Intime,Jenis Faskes Memo,Jenis Faskes Intime,Prov Memo,Prov Intime,Status
0,2,P2171010101,P2171010101,PU930,PU930,Belakang Padang,Belakang Padang,Pustu Lengkang,Pustu Lengkang,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
1,3,P2171010101,P2171010101,PU931,PU931,Belakang Padang,Belakang Padang,Pustu Pulau Sarang,Pustu Pulau Sarang,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
2,4,P2171010101,P2171010101,PU932,PU932,Belakang Padang,Belakang Padang,Pustu Mongkol,Pustu Mongkol,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
3,5,P2171010101,P2171010101,PU933,PU933,Belakang Padang,Belakang Padang,Pustu Pemping,Pustu Pemping,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
4,6,P2171010101,P2171010101,PU934,PU934,Belakang Padang,Belakang Padang,Pustu Pulau Labun,Pustu Pulau Labun,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
5,7,P2171010101,P2171010101,PU935,PU935,Belakang Padang,Belakang Padang,Pustu Kepala Jeri,Pustu Kepala Jeri,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
6,8,P2171010101,P2171010101,PU936,PU936,Belakang Padang,Belakang Padang,Pustu Kasu 1,Pustu Kasu 1,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
7,9,P2171010101,P2171010101,PU937,PU937,Belakang Padang,Belakang Padang,Pustu Kasu 2,Pustu Kasu 2,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
8,10,P2171010101,P2171010101,PU938,PU938,Belakang Padang,Belakang Padang,Pustu Bertam,Pustu Bertam,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama
9,11,P2171010101,P2171010101,PU939,PU939,Belakang Padang,Belakang Padang,Pustu Pecong,Pustu Pecong,Pustu,PUSTU,Kepulauan Riau,Kepulauan Riau,sama


# CHECK DATABASE

In [98]:
# Use URL ID
spreadsheet_server  = '15IMGFofJTm9o1r-yiDRh5i9tC8M4G1qH89szm0cCEUM'
sheet               = gc.open_by_key(spreadsheet_server)
worksheet           = sheet.worksheet("Server ePuskesmas")
data_server         = worksheet.get_all_values()

df_sheet_server = pd.DataFrame(data_server[1:], columns=data_server[0])

df_lookup_server = df_intime.merge(
    df_sheet_server.rename(columns={"puskesmas_id": "ID PELANGGAN"}),
    on="ID PELANGGAN",
    how="left"
)

df_clean_server = df_lookup_server.iloc[:, [0, 1, 2, 3, 4,  5, 6, 22, 23, 24]].copy()
df_clean_server.iloc[:, 5] = df_clean_server.iloc[:, 5].astype(str).str.replace(r'^(pustu|polindes|poskesdes|posyandu|kec)\s+', '', regex=True, flags=re.IGNORECASE).str.strip().str.upper()
df_clean_server.iloc[:, 7] = df_clean_server.iloc[:, 7].str.replace(r'_[^_]+$', '', regex=True)
df_clean_server
# df_lookup_server

,ID PELANGGAN,ID JEJARING,PROVINSI,NAMA KOTA/KAB,NAMA PUSKESMAS,NAMA JEJARING,JENIS FASKES,Database,server,host
0,P2171010101,PU930,Kepulauan Riau,Kota Batam,Belakang Padang,LENGKANG,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
1,P2171010101,PU931,Kepulauan Riau,Kota Batam,Belakang Padang,PULAU SARANG,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
2,P2171010101,PU932,Kepulauan Riau,Kota Batam,Belakang Padang,MONGKOL,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
3,P2171010101,PU933,Kepulauan Riau,Kota Batam,Belakang Padang,PEMPING,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
4,P2171010101,PU934,Kepulauan Riau,Kota Batam,Belakang Padang,PULAU LABUN,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
5,P2171010101,PU935,Kepulauan Riau,Kota Batam,Belakang Padang,KEPALA JERI,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
6,P2171010101,PU936,Kepulauan Riau,Kota Batam,Belakang Padang,KASU 1,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
7,P2171010101,PU937,Kepulauan Riau,Kota Batam,Belakang Padang,KASU 2,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
8,P2171010101,PU938,Kepulauan Riau,Kota Batam,Belakang Padang,BERTAM,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118
9,P2171010101,PU939,Kepulauan Riau,Kota Batam,Belakang Padang,PECONG,PUSTU,epuskesmas_ng_live_kotabatam,kep-riau-21,10.0.131.118


In [99]:
df_to_konfigurasi_ts = pd.DataFrame({
    "Puskesmas ID": df_clean_server["ID PELANGGAN"],
    "Code": df_clean_server["ID PELANGGAN"] + df_clean_server["ID JEJARING"],
    "Nama Jejaring": df_clean_server["NAMA JEJARING"],
    "Jenis Jejaring": df_clean_server["JENIS FASKES"],
    "db_dinkes": df_clean_server["Database"],
    "PIC_TS": PIC,
    "Created At": CREATED_AT
})
    
# display(df_to_konfigurasi_ts)

In [100]:
# Configure db
username = "root"
password = "root"           
host     = "localhost"
database = "dummy_data"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# Check connection
with engine.connect() as conn:
    print("Connected!")

Connected!


In [ ]:
# REPORT 3a : CEK PUSKESMAS

# Ambil ID Puskesmas unik
id_puskesmas_list = (
    df_to_konfigurasi_ts["Puskesmas ID"]
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

# print(id_puskesmas_list)

id_puskesmas_sql = ",".join(f"'{x}'" for x in id_puskesmas_list)

query_puskesmas = f"""
SELECT id AS puskesmas_id
FROM m_puskesmas
WHERE id IN ({id_puskesmas_sql});
"""

df_puskesmas_db = pd.read_sql(query_puskesmas, con=engine)
puskesmas_ada   = set(df_puskesmas_db["puskesmas_id"].astype(str))

rows_3a = []

for pid in id_puskesmas_list: 
    status = (
        "OK - FASKES SUDAH ADA"
        if pid in puskesmas_ada
        else "FASKES BELUM ADA"
    )

    rows_3a.append({
        "Puskesmas ID": pid,
        "Status": status
    })

df_report_3a = pd.DataFrame(rows_3a)

print("REPORT 3a \u2014 VALIDASI FASKES (PUSKESMAS)")
display(df_report_3a)


REPORT 3a — VALIDASI FASKES (PUSKESMAS)


,Puskesmas ID,Status
0,P2171010101,FASKES BELUM ADA
1,P2171020201,FASKES BELUM ADA
2,P2171030201,FASKES BELUM ADA
3,P2171030202,FASKES BELUM ADA
4,P2171040101,FASKES BELUM ADA
5,P2171050201,FASKES BELUM ADA
6,P2171050202,FASKES BELUM ADA
7,P2171050203,FASKES BELUM ADA
8,P2171050204,FASKES BELUM ADA
9,P2171051201,FASKES BELUM ADA


In [ ]:
# REPORT 3b : CEK JEJARING 

rows_3b = []

def norm(val):
    if pd.isna(val):
        return ""
    return str(val).strip().upper()


for _, row in df_to_konfigurasi_ts.iterrows():

    id_jejaring    = norm(row["Code"])
    nama_jejaring  = norm(row["Nama Jejaring"])
    jenis_jejaring = norm(row["Jenis Jejaring"])


    # CEK ID JEJARING
    q_id = f"""
    SELECT 1 FROM m_puskesmas
    WHERE id = '{id_jejaring}'
    LIMIT 1;
    """
    id_exist = not pd.read_sql(q_id, con=engine).empty

    # CEK NAMA + JENIS JEJARING
    q_nama_jenis = f"""
    SELECT 1 FROM m_puskesmas
    WHERE UPPER(nama) = '{nama_jejaring}' AND UPPER(jenis_puskesmas) = '{jenis_jejaring}'
    LIMIT 1;
    """
    nama_jenis_exist = not pd.read_sql(q_nama_jenis, con=engine).empty

 
    # LOGIC STATUS
    if id_exist:
        status = "ID JEJARING SUDAH ADA"

    elif (not id_exist) and nama_jenis_exist:
        status = "NAMA DAN JENIS JEJARING SUDAH ADA"

    else:
        status = "OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)"

    rows_3b.append({
        "ID Jejaring"   : id_jejaring,
        "Nama Jejaring" : row["Nama Jejaring"],
        "Jenis Jejaring": row["Jenis Jejaring"],
        "Status": status
    })

df_report_3b = pd.DataFrame(rows_3b)

print("REPORT 3b \u2014 VALIDASI JEJARING")
display(df_report_3b)


REPORT 3b — VALIDASI JEJARING


,ID Jejaring,Nama Jejaring,Jenis Jejaring,Status
0,P2171010101PU930,LENGKANG,PUSTU,ID JEJARING SUDAH ADA
1,P2171010101PU931,PULAU SARANG,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
2,P2171010101PU932,MONGKOL,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
3,P2171010101PU933,PEMPING,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
4,P2171010101PU934,PULAU LABUN,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
5,P2171010101PU935,KEPALA JERI,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
6,P2171010101PU936,KASU 1,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
7,P2171010101PU937,KASU 2,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
8,P2171010101PU938,BERTAM,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)
9,P2171010101PU939,PECONG,PUSTU,OK - JEJARING BELUM ADA (SIAP DIAKTIVASI)


In [ ]:
LOOKUP_MASTER_DB = "dummy_data"   # lokal
# LOOKUP_MASTER_DB = "epuskesmas_master"  # production

def norm(val):
    if pd.isna(val):
        return ""
    return str(val).strip().upper()

rows_report4 = []

for _, row in df_clean_server.iterrows():

    jenis_faskes = norm(row["JENIS FASKES"])
    id_jejaring  = row["ID JEJARING"]

    if jenis_faskes == "":
        rows_report4.append({
            "ID Jejaring": id_jejaring,
            "Jenis Faskes": jenis_faskes,
            "Jumlah di m_lookup": None,
            "Status": "JENIS FASKES KOSONG"
        })
        continue

    query = f"""
    SELECT COUNT(1) AS total
    FROM {LOOKUP_MASTER_DB}.m_lookup
    WHERE `for` = 'jenis_puskesmas'
      AND UPPER(`value`) = '{jenis_faskes}';
    """

    try:
        total = int(pd.read_sql(query, con=engine).iloc[0, 0])
        status = (
            "JENIS JEJARING SUDAH ADA"
            if total > 0
            else "JENIS JEJARING TIDAK ADA (PERLU INSERT)"
        )
    except Exception:
        total   = None
        status  = "ERROR LOOKUP MASTER"

    rows_report4.append({
        "ID Jejaring": id_jejaring,
        "Jenis Faskes": jenis_faskes,
        "Jumlah di m_lookup": total,
        "Status": status
    })

df_report4 = pd.DataFrame(rows_report4)

print("REPORT 4 — VALIDASI JENIS JEJARING (MASTER LOOKUP)")
display(df_report4)


REPORT 4 — VALIDASI JENIS JEJARING (MASTER LOOKUP)


,ID Jejaring,Jenis Faskes,Jumlah di m_lookup,Status
0,PU930,PUSTU,1,JENIS JEJARING SUDAH ADA
1,PU931,PUSTU,1,JENIS JEJARING SUDAH ADA
2,PU932,PUSTU,1,JENIS JEJARING SUDAH ADA
3,PU933,PUSTU,1,JENIS JEJARING SUDAH ADA
4,PU934,PUSTU,1,JENIS JEJARING SUDAH ADA
5,PU935,PUSTU,1,JENIS JEJARING SUDAH ADA
6,PU936,PUSTU,1,JENIS JEJARING SUDAH ADA
7,PU937,PUSTU,1,JENIS JEJARING SUDAH ADA
8,PU938,PUSTU,1,JENIS JEJARING SUDAH ADA
9,PU939,PUSTU,1,JENIS JEJARING SUDAH ADA


In [ ]:
# REPORT 4 : CEK JENIS JEJARING DI m_lookup

rows_report4 = []

for _, row in df_clean_server.iterrows():

    database     = str(row["Database"]).strip()
    jenis_faskes = norm(row["JENIS FASKES"])
    id_jejaring  = row["ID JEJARING"]


    # Validasi awal
    if database == "" or jenis_faskes == "":
        rows_report4.append({
            "Database": database,
            "ID Jejaring": id_jejaring,
            "Jenis Faskes": jenis_faskes,
            "Jumlah di m_lookup": None,
            "Status": "DATABASE / JENIS FASKES KOSONG"
        })
        continue

 
    # Query cek jenis_puskesmas
    query = f"""
    SELECT COUNT(1) AS total, '{jenis_faskes}' AS value
    FROM {database}.m_lookup
    WHERE `for` = 'jenis_puskesmas'
      AND UPPER(`value`) = '{jenis_faskes}';
    """

    try:
        df_check = pd.read_sql(query, con=engine)
        total = int(df_check.iloc[0]["total"])

        if total > 0:
            status = "JENIS JEJARING SUDAH ADA"
        else:
            status = "JENIS JEJARING TIDAK ADA (PERLU INSERT)"

    except Exception as e:
        total = None
        status = f"ERROR QUERY: {str(e)}"

    rows_report4.append({
        "Database": database,
        "ID Jejaring": id_jejaring,
        "Jenis Faskes": jenis_faskes,
        "Jumlah di m_lookup": total,
        "Status": status
    })


df_report4 = pd.DataFrame(rows_report4)
print("REPORT 4 \u2014 VALIDASI JENIS JEJARING (m_lookup)")
display(df_report4)


REPORT 4 — VALIDASI JENIS JEJARING (m_lookup)


,Database,ID Jejaring,Jenis Faskes,Jumlah di m_lookup,Status
0,epuskesmas_ng_live_kotabatam,PU930,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
1,epuskesmas_ng_live_kotabatam,PU931,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
2,epuskesmas_ng_live_kotabatam,PU932,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
3,epuskesmas_ng_live_kotabatam,PU933,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
4,epuskesmas_ng_live_kotabatam,PU934,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
5,epuskesmas_ng_live_kotabatam,PU935,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
6,epuskesmas_ng_live_kotabatam,PU936,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
7,epuskesmas_ng_live_kotabatam,PU937,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
8,epuskesmas_ng_live_kotabatam,PU938,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sqlalche.me/e/20/f405)"
9,epuskesmas_ng_live_kotabatam,PU939,PUSTU,None,"ERROR QUERY: (pymysql.err.ProgrammingError) (1146, ""Table 'epuskesmas_ng_live_kotabatam.m_lookup' doesn't exist"")\n[SQL: \n SELECT COUNT(1) AS total, 'PUSTU' AS value\n FROM epuskesmas_ng_live_kotabatam.m_lookup\n WHERE `for` = 'jenis_puskesmas'\n AND UPPER(`value`) = 'PUSTU';\n ]\n(Background on this error at: https://sql

In [20]:
id_list = df_to_konfigurasi_ts["Code"].astype(str).unique().tolist()
id_list_sql = ",".join(f"'{x}'" for x in id_list)

query = f"""
SELECT 
    id,
    jenis_puskesmas,
    nama,
    db_database,
    alamat,
    stok_obat_realtime,
    created_at,
    updated_at
FROM m_puskesmas
WHERE id IN ({id_list_sql});
"""

df_lookup = pd.read_sql(query, con=engine)

df_merged = df_to_konfigurasi_ts.merge(
    df_lookup,
    left_on="Code",
    right_on="id",
    how="left"
)

df_merged["Status DB"] = df_merged["id"].apply(
    lambda x: "ADA DI DATABASE" if pd.notnull(x) else "TIDAK ADA"
)

print('Berikut hasil pencocokan jejaring dengan database :')
# df_merged = df_merged.iloc[:, [1, 2, 3, 4, 7, 15]].copy()
display(df_merged)

Berikut hasil pencocokan jejaring dengan database :


,Puskesmas ID,Code,Nama Jejaring,Jenis Jejaring,db_dinkes,PIC_TS,Created At,id,jenis_puskesmas,nama,db_database,alamat,stok_obat_realtime,created_at,updated_at,Status DB
0,P7212050101,P7212050101PU001,GILILANA,PUSTU,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
1,P7212050101,P7212050101PU002,⁠TANAUGE,PUSTU,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
2,P7212050101,P7212050101PU003,GANDA GANDA,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
3,P7212050101,P7212050101PU004,KOROMATANTU,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
4,P7212050101,P7212050101PU005,⁠KOROLOLAKI,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
5,P7212050101,P7212050101PU006,KOYA,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
6,P7212050101,P7212050101PU007,KOROLOLAMA,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
7,P7212050101,P7212050101PU008,BAHONTULA,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
8,P7212050101,P7212050101PU009,BAHOUE,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA
9,P7212050101,P7212050101PU010,KOLONODALE,POSKESDES,epuskesmas_ng_live_morowaliutara,ts_iqbal_agus,2025-12-28 13:16:48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TIDAK ADA


In [ ]:
# ======================================================
# REPORT 3 (SUPSERSET)
# 3a: VALIDASI ID PELANGGAN (WAJIB SUDAH ADA)
# 3b: VALIDASI ID JEJARING (WAJIB BELUM ADA)
# ======================================================

import pandas as pd

rows_report3 = []

# ======================================================
# STEP 1 — LOOKUP ID PELANGGAN (SUPSERSET DB)
# ======================================================

id_pelanggan_list = (df_to_konfigurasi_ts["Puskesmas ID"].astype(str).str.strip().unique().tolist())

id_pelanggan_sql = ",".join(f"'{x}'" for x in id_pelanggan_list)

query_pelanggan = f"""
SELECT id
FROM m_puskesmas
WHERE id IN ({id_pelanggan_sql});
"""

df_pelanggan_db = pd.read_sql(query_pelanggan, con=engine)
id_pelanggan_ada = set(df_pelanggan_db["id"].astype(str))

# ======================================================
# STEP 2 — LOOP REPORT 3
# ======================================================
for _, row in df_to_konfigurasi_ts.iterrows():

    id_pelanggan   = str(row["Puskesmas ID"]).strip()
    id_jejaring    = str(row["Code"]).strip()   # atau kolom ID JEJARING
    nama_jejaring  = row["Nama Jejaring"]
    jenis_jejaring = row["Jenis Jejaring"]

    # --------------------------------------------------
    # REPORT 3a — CEK ID PELANGGAN
    # --------------------------------------------------
    if id_pelanggan not in id_pelanggan_ada:
        status = "ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)"

    else:
        # --------------------------------------------------
        # REPORT 3b — CEK ID JEJARING
        # --------------------------------------------------
        query_jejaring = f"""
        SELECT id
        FROM m_puskesmas
        WHERE id = '{id_jejaring}';
        """
        df_jejaring_db = pd.read_sql(query_jejaring, con=engine)

        if not df_jejaring_db.empty:
            status = "ERROR 3B: ID JEJARING SUDAH ADA (AKUN SUDAH DIBUAT)"
        else:
            status = "OK: SIAP AKTIVASI AKUN JEJARING"

    rows_report3.append({
        "ID PELANGGAN": id_pelanggan,
        "ID JEJARING": id_jejaring,
        "NAMA JEJARING": nama_jejaring,
        "JENIS JEJARING": jenis_jejaring,
        "STATUS": status
    })

# ======================================================
# OUTPUT REPORT 3
# ======================================================
df_report3 = pd.DataFrame(rows_report3)

print("REPORT 3 (SUPSERSET) — VALIDASI AKUN JEJARING")
display(df_report3)


REPORT 3 (SUPSERSET) — VALIDASI AKUN JEJARING


,ID PELANGGAN,ID JEJARING,NAMA JEJARING,JENIS JEJARING,STATUS
0,P7212050101,P7212050101PU001,GILILANA,PUSTU,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
1,P7212050101,P7212050101PU002,⁠TANAUGE,PUSTU,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
2,P7212050101,P7212050101PU003,GANDA GANDA,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
3,P7212050101,P7212050101PU004,KOROMATANTU,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
4,P7212050101,P7212050101PU005,⁠KOROLOLAKI,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
5,P7212050101,P7212050101PU006,KOYA,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
6,P7212050101,P7212050101PU007,KOROLOLAMA,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
7,P7212050101,P7212050101PU008,BAHONTULA,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
8,P7212050101,P7212050101PU009,BAHOUE,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)
9,P7212050101,P7212050101PU010,KOLONODALE,POSKESDES,ERROR 3A: ID PELANGGAN BELUM ADA (AKUN INDUK BELUM DIBUAT)


☑️MENYALIN DAN MELENGKAPI INFORMASI JEJARING DARI INTIME KE KONFIGURASI TS☑️

In [52]:
SPREADSHEET_KONFIGURASI_TS  = '1IAR5thTwLHs42eJ28sqJMPbslhHIm8HnJqH7iTVhDmc'
SHEET_NAME_TARGET           = "Jejaring"

# Get the number of filled rows in the sheet
sheet_metadata = service.spreadsheets().get(spreadsheetId=SPREADSHEET_KONFIGURASI_TS).execute()
sheet_props = next(
    (s for s in sheet_metadata.get("sheets", []) if s["properties"]["title"] == SHEET_NAME_TARGET),
    None
)

if sheet_props:
    grid_props = sheet_props["properties"]["gridProperties"]
    read_range = f"{SHEET_NAME_TARGET}!A:A"
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_KONFIGURASI_TS, range=read_range
    ).execute()
    existing_rows = len(result.get("values", []))
else:
    raise Exception(f"Sheet '{SHEET_NAME_TARGET}' tidak ditemukan.")

# Specify the starting line for appending
start_row = existing_rows + 1
range_append = f"{SHEET_NAME_TARGET}!A{start_row}"
end_row = start_row + len(df_to_konfigurasi_ts) - 1
values = df_to_konfigurasi_ts.values.tolist()
# Append data
service.spreadsheets().values().append(
    spreadsheetId=SPREADSHEET_KONFIGURASI_TS,
    range=range_append,
    valueInputOption="RAW",
    insertDataOption="INSERT_ROWS",
    body={"values": values}
).execute()

print(f"{len(values)} baris telah ditambahkan dari baris {start_row} sampai baris {end_row}.")

11 baris telah ditambahkan dari baris 14 sampai baris 24.


In [ ]:
# ==============================================================================================================================================================================================================

In [ ]:
# # cleansing data from sheet Detail APJ for sheet Jejaring
df_selected         = df_lookup_server.iloc[:, [0, 1, 5, 6, 2, 3]]
df_selected.columns = ['ID PELANGGAN', 'ID JEJARING','NAMA JEJARING', 'JENIS FASKES', 'PROVINSI', 'NAMA KOTA/KAB']
df_selected['NAMA JEJARING']  = (df_selected['NAMA JEJARING'].astype(str).str.replace(r'^(pustu|polindes|poskesdes|posyandu|kec)\s+', '', regex=True, flags=re.IGNORECASE).str.strip().str.upper())
df_selected['NAMA SERVER']    = ('epuskesmas_ng_live_' + df_selected['NAMA KOTA/KAB'].astype(str).str.lower().str.replace(r'\b(kabupaten|kab|kota)\b', '', regex=True).str.replace(r'[^a-z0-9]', '', regex=True))
df_selected

/tmp/ipykernel_19369/993084553.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['NAMA JEJARING']  = (df_selected['NAMA JEJARING'].astype(str).str.replace(r'^(pustu|polindes|poskesdes|posyandu|kec)\s+', '', regex=True, flags=re.IGNORECASE).str.strip().str.upper())
/tmp/ipykernel_19369/993084553.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['NAMA SERVER']    = ('epuskesmas_ng_live_' + df_selected['NAMA KOTA/KAB'].astype(str).str.lower().str.replace(r'\b(kabupaten|kab|kota)\b',

,ID PELANGGAN,ID JEJARING,NAMA JEJARING,JENIS FASKES,PROVINSI,NAMA KOTA/KAB,NAMA SERVER
26,P2171010101,PU930,LENGKANG,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
27,P2171010101,PU931,PULAU SARANG,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
28,P2171010101,PU932,MONGKOL,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
29,P2171010101,PU933,PEMPING,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
30,P2171010101,PU934,PULAU LABUN,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
31,P2171010101,PU935,KEPALA JERI,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
32,P2171010101,PU936,KASU 1,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
33,P2171010101,PU937,KASU 2,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
34,P2171010101,PU938,BERTAM,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam
35,P2171010101,PU939,PECONG,PUSTU,Kepulauan Riau,Kota Batam,epuskesmas_ng_live_batam


In [28]:
df_to_append = pd.DataFrame({
    "Puskesmas ID": df_selected["ID PELANGGAN"],
    "Code": df_selected["ID PELANGGAN"] + df_selected["ID JEJARING"],
    "Nama Jejaring": df_selected["NAMA JEJARING"],
    "Jenis Jejaring": df_selected["JENIS FASKES"],
    "db_dinkes": df_selected["NAMA SERVER"],
    "PIC_TS": PIC,
    "Created At": CREATED_AT
})

display(df_to_append)

# cross check value to data frame list
# values = df_to_append.values.tolist()
# values

,Puskesmas ID,Code,Nama Jejaring,Jenis Jejaring,db_dinkes,PIC_TS,Created At
11,P7318010101,P7318010101PU001,POTON,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
12,P7318010101,P7318010101PU002,NUSA,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
13,P7318010101,P7318010101PU003,SALUBARANA,POLINDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
14,P7318010101,P7318010101PU004,BAU,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
15,P7318010101,P7318010101PU005,PEAUN,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
16,P7318010101,P7318010101PU006,MAPPA,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
17,P7318051102,P7318051102PU001,SATANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
18,P7318051102,P7318051102PU002,SALUTANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
19,P7318051102,P7318051102PU003,SALU,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54
20,P7318051102,P7318051102PU004,SALU BORONAN,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-26 15:19:54


In [29]:
SPREADSHEET_KONFIGURASI_TS  = '1IAR5thTwLHs42eJ28sqJMPbslhHIm8HnJqH7iTVhDmc'
SHEET_NAME_TARGET           = "Jejaring"

# Get the number of filled rows in the sheet
sheet_metadata = service.spreadsheets().get(spreadsheetId=SPREADSHEET_KONFIGURASI_TS).execute()
sheet_props = next(
    (s for s in sheet_metadata.get("sheets", []) if s["properties"]["title"] == SHEET_NAME_TARGET),
    None
)

if sheet_props:
    grid_props = sheet_props["properties"]["gridProperties"]
    read_range = f"{SHEET_NAME_TARGET}!A:A"
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_KONFIGURASI_TS, range=read_range
    ).execute()
    existing_rows = len(result.get("values", []))
else:
    raise Exception(f"Sheet '{SHEET_NAME_TARGET}' tidak ditemukan.")

# Specify the starting line for appending
start_row = existing_rows + 1
range_append = f"{SHEET_NAME_TARGET}!A{start_row}"
end_row = start_row + len(df_selected) - 1
values = df_to_append.values.tolist()
# Append data
service.spreadsheets().values().append(
    spreadsheetId=SPREADSHEET_KONFIGURASI_TS,
    range=range_append,
    valueInputOption="RAW",
    insertDataOption="INSERT_ROWS",
    body={"values": values}
).execute()

print(f"{len(values)} row successfully added starting from row {start_row} until {end_row}.")

15 row successfully added starting from row 29 until 43.


🔴CONNECT DATABASE🔴

In [33]:
# Configure db
username = "root"
password = "root"           
host     = "localhost"
database = "dummy_data"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# Check connection
with engine.connect() as conn:
    print("Connected!")

Connected!


🔴CEK JEJARING DI SERVER🔴

In [ ]:
def generate_select_lookup(row):
    id = 'P7318010101'  # id yang ingin dicek

    query = (
        f"SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, "
        f"created_at, updated_at FROM m_puskesmas WHERE id = '{id}'"
    )
    return query

# Tidak perlu loop, cukup 1 query
final_query = generate_select_lookup(None)
# print(final_query)

try:
    df_result = pd.read_sql(final_query, con=engine)
    print("\nHASIL QUERY:")
    # print(df_result)
    
    if df_result.empty:
        print("⚠️ ID TIDAK ADA DI DATABASE")
    else:
        print("✅ ID TERDAFTAR DI DATABASE")

except Exception as e:
    print("Error:", e)



HASIL QUERY:
✅ ID TERDAFTAR DI DATABASE


🔴GET DATA SERVER🔴

In [30]:
# Get faskes id unique for server
id_faskes = df_selected["ID PELANGGAN"].unique()

# Use URL ID
spreadsheet_server  = '15IMGFofJTm9o1r-yiDRh5i9tC8M4G1qH89szm0cCEUM'
sheet               = gc.open_by_key(spreadsheet_server)
worksheet           = sheet.worksheet("Server ePuskesmas")
data_server         = worksheet.get_all_values()

df_sheet_server = pd.DataFrame(data_server[1:], columns=data_server[0])
df_server       = pd.DataFrame()

for server_faskes in id_faskes:
    df_filtered = df_sheet_server[df_sheet_server["puskesmas_id"] == server_faskes]
    df_server   = pd.concat([df_server, df_filtered], ignore_index=True)

    # print(f"Hasil untuk {server_faskes}:")
    # print(df_filtered)

df_server

,puskesmas_id,nama_faskes,Database,server,host
0,P7318010101,BUAKAYU,epuskesmas_ng_live_tanatoraja_P7318010101,sulawesi-selatan-73,10.0.131.208
1,P7318051102,ULUSALU,epuskesmas_ng_live_tanatoraja_P7318051102,sulawesi-selatan-73,10.0.131.208
2,P7318021102,BUNTU,epuskesmas_ng_live_tanatoraja_P7318021102,sulawesi-selatan-73,10.0.131.208


🔴CONNECT DATABASE🔴

In [31]:
# Configure db
username = "root"
password = "root"           
host     = "localhost"
database = "dummy_data"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# Check connection
with engine.connect() as conn:
    print("Connected!")

Connected!


In [32]:
def generate_select_lookup(row):
    # id             = row['ID PELANGGAN']
    id             = str('P7318010101')


    query = (
        f"SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = {id}"
    )
    return query

queries     = [generate_select_lookup(row) for _, row in df_to_append.iterrows()]
final_query = " UNION ALL\n".join(queries) + ";"
print(final_query)

try:
    df_result = pd.read_sql(final_query, con=engine)
    print("\nHASIL QUERY:")
    print(df_result)

except Exception as e:
    print("Error:", e)

SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = P7318010101 UNION ALL
SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesm

In [19]:
df_to_append = pd.DataFrame({
    "ID PELANGGAN": df_selected["ID PELANGGAN"],
    "Code": df_selected["ID PELANGGAN"] + df_selected["ID JEJARING"],
    "Nama Jejaring": df_selected["NAMA JEJARING"],
    "Jenis Jejaring": df_selected["JENIS FASKES"],
    "db_dinkes": df_selected["NAMA SERVER"],
    "PIC_TS": PIC,
    "Created At": CREATED_AT
})

display(df_to_append)

,ID PELANGGAN,Code,Nama Jejaring,Jenis Jejaring,db_dinkes,PIC_TS,Created At
11,P7318010101,P7318010101PU001,POTON,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
12,P7318010101,P7318010101PU002,NUSA,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
13,P7318010101,P7318010101PU003,SALUBARANA,POLINDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
14,P7318010101,P7318010101PU004,BAU,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
15,P7318010101,P7318010101PU005,PEAUN,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
16,P7318010101,P7318010101PU006,MAPPA,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
17,P7318051102,P7318051102PU001,SATANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
18,P7318051102,P7318051102PU002,SALUTANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
19,P7318051102,P7318051102PU003,SALU,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00
20,P7318051102,P7318051102PU004,SALU BORONAN,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-24 12:00:00


In [ ]:
def generate_select_lookup(row):
    # id             = row['ID PELANGGAN']
    id             = str('P7318010101')


    query = (
        f"SELECT id, jenis_puskesmas, nama, db_database, alamat, stok_obat_realtime, created_at, updated_at FROM m_puskesmas WHERE id = {id}"
    )
    return query

queries     = [generate_select_lookup(row) for _, row in df_to_append.iterrows()]
final_query = " UNION ALL\n".join(queries) + ";"
print(final_query)

try:
    df_result = pd.read_sql(final_query, con=engine)
    print("\nHASIL QUERY:")
    print(df_result)

except Exception as e:
    print("Error:", e)

INSERT KE SERVER

In [41]:
from sqlalchemy import text

# Daftar jejaring (input manual)
jejaring_list = [
    {
        "id_induk": "P7318010101",
        "kode": "PU001",
        "jenis": "PUSTU",
        "nama": "GILILANA"
    },
    {
        "id_induk": "P7318010101",
        "kode": "PU002",
        "jenis": "PUSTU",
        "nama": "TANAUGE"
    },
    # tambah jika perlu
]


def generate_manual_query(item):
    id_induk = item["id_induk"]
    code     = f"{id_induk}{item['kode']}"
    jenis    = item["jenis"]
    nama     = item["nama"]

    return (
        f"SELECT '{code}' AS id, '{jenis}' AS jenis_puskesmas, '{nama}' AS nama, "
        f"db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id AS puskesmas_id, "
        f"0 AS warning_text_id, 1 AS stok_obat_realtime "
        f"FROM m_puskesmas WHERE id = '{id_induk}'"
    )


# Build final INSERT ... SELECT
queries = [generate_manual_query(item) for item in jejaring_list]

final_query = (
    "INSERT INTO dummy_data.`m_puskesmas` "
    "(`id`, `jenis_puskesmas`, `nama`, `db_database`, `propinsi_id`, `kota_id`, `kecamatan_id`, `tanggal_tunda`, "
    "`puskesmas_id`, `warning_text_id`, `stok_obat_realtime`)\n"
    + "UNION ALL\n".join(queries)
    + ";"
)

print("QUERY AKHIR:\n")
print(final_query)


# \U0001f525 EKSEKUSI QUERY INSERT
try:
    with engine.begin() as conn:
        conn.execute(text(final_query))
    print("\n\u2714 DATA BERHASIL DIINSERT KE m_puskesmas!")

except Exception as e:
    print("\n\u274c Error eksekusi:", e)

QUERY AKHIR:

INSERT INTO dummy_data.`m_puskesmas` (`id`, `jenis_puskesmas`, `nama`, `db_database`, `propinsi_id`, `kota_id`, `kecamatan_id`, `tanggal_tunda`, `puskesmas_id`, `warning_text_id`, `stok_obat_realtime`)
SELECT 'P7318010101PU001' AS id, 'PUSTU' AS jenis_puskesmas, 'GILILANA' AS nama, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id AS puskesmas_id, 0 AS warning_text_id, 1 AS stok_obat_realtime FROM m_puskesmas WHERE id = 'P7318010101'UNION ALL
SELECT 'P7318010101PU002' AS id, 'PUSTU' AS jenis_puskesmas, 'TANAUGE' AS nama, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id AS puskesmas_id, 0 AS warning_text_id, 1 AS stok_obat_realtime FROM m_puskesmas WHERE id = 'P7318010101';

✔ DATA BERHASIL DIINSERT KE m_puskesmas!


In [22]:
def generate_select_lookup(row):
    db             = row['db_dinkes']
    jenis_jejaring = row['Jenis Jejaring']

    query = (
        f"SELECT count(1), `value` FROM {db}.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = '{jenis_jejaring}' UNION ALL"
    )
    return query

queries     = [generate_select_lookup(row) for _, row in df_to_append.iterrows()]
final_query = " UNION ALL\n".join(queries) + ";"
# print(final_query)

try:
    df_result = pd.read_sql(final_query, con=engine)
    print("\nHASIL QUERY:")
    print(df_result)

except Exception as e:
    print("Error:", e)

Error: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near '`for` = 'jenis_puskesmas' and `value` = 'PUSTU' UNION ALL UNION ALL\nSELECT co...' at line 1")
[SQL: SELECT count(1), `value` FROM epuskesmas_ng_live_tanatoraja.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = 'PUSTU' UNION ALL UNION ALL
SELECT count(1), `value` FROM epuskesmas_ng_live_tanatoraja.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = 'PUSTU' UNION ALL UNION ALL
SELECT count(1), `value` FROM epuskesmas_ng_live_tanatoraja.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = 'POLINDES' UNION ALL UNION ALL
SELECT count(1), `value` FROM epuskesmas_ng_live_tanatoraja.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = 'POSKESDES' UNION ALL UNION ALL
SELECT count(1), `value` FROM epuskesmas_ng_live_tanatoraja.m_lookup HERE `for` = 'jenis_puskesmas' and `value` = 'POSKESDES' UNION ALL UNION

🔴SAVE TO DATAFRAME🔴

In [ ]:
df_to_append = pd.DataFrame({
    "Puskesmas ID": df_selected["ID PELANGGAN"],
    "Code": df_selected["ID PELANGGAN"] + df_selected["ID JEJARING"],
    "Nama Jejaring": df_selected["NAMA JEJARING"],
    "Jenis Jejaring": df_selected["JENIS FASKES"],
    "db_dinkes": df_selected["NAMA SERVER"],
    "PIC_TS": PIC,
    "Created At": CREATED_AT
})

display(df_to_append)

# cross check value to data frame list
# values = df_to_append.values.tolist()
# values

,Puskesmas ID,Code,Nama Jejaring,Jenis Jejaring,db_dinkes,PIC_TS,Created At
11,P7318010101,P7318010101PU001,POTON,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
12,P7318010101,P7318010101PU002,NUSA,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
13,P7318010101,P7318010101PU003,SALUBARANA,POLINDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
14,P7318010101,P7318010101PU004,BAU,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
15,P7318010101,P7318010101PU005,PEAUN,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
16,P7318010101,P7318010101PU006,MAPPA,POSKESDES,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
17,P7318051102,P7318051102PU001,SATANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
18,P7318051102,P7318051102PU002,SALUTANDUNG,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
19,P7318051102,P7318051102PU003,SALU,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00
20,P7318051102,P7318051102PU004,SALU BORONAN,PUSTU,epuskesmas_ng_live_tanatoraja,ts_iqbal_agus,2025-11-17 12:00:00


# INSERT FASKES DAN DINKES

In [ ]:
SPREADSHEET_KONFIGURASI_TS  = '1IAR5thTwLHs42eJ28sqJMPbslhHIm8HnJqH7iTVhDmc'
SHEET_NAME_TARGET           = "Jejaring"

INSERT DINKES

In [ ]:
def generate_insert_dinkes(row):
    db_dinkes      = row['db_dinkes']
    code           = row['Code']
    jenis_jejaring = row['Jenis Jejaring']
    nama_jejaring  = row['Nama Jejaring']
    puskesmas_id   = row['Puskesmas ID']
    created_at     = row['Created At']
    pic            = row['PIC_TS']

    query = (
        f"SELECT '{code}' AS code, '{jenis_jejaring}' AS jenis_jejaring,'{nama_jejaring}' AS nama_jejaring, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id, 0 AS WT, 1 AS STOK, '{created_at}' AS created_at, '{created_at}' AS updated_at, '{pic}' AS created_by, '{pic}' AS updated_by FROM m_puskesmas WHERE id = '{puskesmas_id}'"
    )
    return query

queries     = [generate_insert_dinkes(row) for _, row in df_to_append.iterrows()]
final_query = " UNION ALL\n".join(queries) + ";"
print(final_query)

SELECT 'P7318010101PU001' AS code, 'PUSTU' AS jenis_jejaring,'POTON' AS nama_jejaring, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id, 0 AS WT, 1 AS STOK, '2025-11-14 12:00:00' AS created_at, '2025-11-14 12:00:00' AS updated_at, 'ts_iqbal_agus' AS created_by, 'ts_iqbal_agus' AS updated_by FROM m_puskesmas WHERE id = 'P7318010101' UNION ALL
SELECT 'P7318010101PU002' AS code, 'PUSTU' AS jenis_jejaring,'NUSA' AS nama_jejaring, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id, 0 AS WT, 1 AS STOK, '2025-11-14 12:00:00' AS created_at, '2025-11-14 12:00:00' AS updated_at, 'ts_iqbal_agus' AS created_by, 'ts_iqbal_agus' AS updated_by FROM m_puskesmas WHERE id = 'P7318010101' UNION ALL
SELECT 'P7318010101PU003' AS code, 'POLINDES' AS jenis_jejaring,'SALUBARANA' AS nama_jejaring, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id, 0 AS WT, 1 AS STOK, '2025-11-14 12:00:00' AS created_at, '2025-11-14 12:00:00' AS updated_at, 'ts_iqbal_agus' AS c

In [ ]:
def generate_insert_dinkes(row):
    db_dinkes      = row['db_dinkes']
    code           = row['Code']
    jenis_jejaring = row['Jenis Jejaring']
    nama_jejaring  = row['Nama Jejaring']
    puskesmas_id   = row['Puskesmas ID']
    created_at     = row['Created At']
    pic            = row['PIC_TS']

    query = (
        f"SELECT '{code}' AS code, '{jenis_jejaring}' AS jenis_jejaring,'{nama_jejaring}' AS nama_jejaring, db_database, propinsi_id, kota_id, kecamatan_id, tanggal_tunda, id, 0 AS WT, 1 AS STOK, '{created_at}' AS created_at, '{created_at}' AS updated_at, '{pic}' AS created_by, '{pic}' AS updated_by FROM m_puskesmas WHERE id = '{puskesmas_id}'"
    )
    return query

queries = [generate_insert_dinkes(row) for _, row in df_to_append.iterrows()]

final_query = " UNION ALL\n".join(queries) + ";"

print(final_query)

In [ ]:
# Get the number of filled rows in the sheet
sheet_metadata = service.spreadsheets().get(spreadsheetId=SPREADSHEET_KONFIGURASI_TS).execute()
sheet_props = next(
    (s for s in sheet_metadata.get("sheets", []) if s["properties"]["title"] == SHEET_NAME_TARGET),
    None
)

if sheet_props:
    grid_props = sheet_props["properties"]["gridProperties"]
    read_range = f"{SHEET_NAME_TARGET}!A:A"
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_KONFIGURASI_TS, range=read_range
    ).execute()
    existing_rows = len(result.get("values", []))
else:
    raise Exception(f"Sheet '{SHEET_NAME_TARGET}' tidak ditemukan.")

# Specify the starting line for appending
start_row = existing_rows + 1
range_append = f"{SHEET_NAME_TARGET}!A{start_row}"
end_row = start_row + len(df_selected) - 1
values = df_to_append.values.tolist()
# Append data
service.spreadsheets().values().append(
    spreadsheetId=SPREADSHEET_KONFIGURASI_TS,
    range=range_append,
    valueInputOption="RAW",
    insertDataOption="INSERT_ROWS",
    body={"values": values}
).execute()

print(f"{len(values)} row successfully added starting from row {start_row} until {end_row}.")

15 row successfully added starting from row 14 until 28.
